# 01 — Análise de Distribuição do Dataset
**NurseXAI** | Pré-processamento e análise exploratória

---
**Executar antes de:** `02_model_training.ipynb`

Este notebook realiza:
1. Carregamento e validação do dataset
2. Binarização de labels multi-label
3. Análise de distribuição (frequência, label density, co-ocorrência)
4. Geração das figuras para o artigo

## 1. Setup e Imports

In [ ]:
# Instalar dependências (apenas na primeira execução)
# !pip install -r ../requirements.txt -q

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from sklearn.preprocessing import MultiLabelBinarizer

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

print("✅ Imports OK")
print(f"   numpy  : {np.__version__}")
print(f"   pandas : {pd.__version__}")

## 2. Carregamento do Dataset

In [ ]:
# Google Colab: DATA_PATH = '/content/notas_dataset_completo.csv'
# Local:        DATA_PATH = '../data/notas_dataset_completo.csv'

DATA_PATH = '../data/notas_dataset_completo.csv'  # ajustar se necessário

df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')

print(f"✅ Dataset carregado")
print(f"   Shape   : {df.shape}")
print(f"   Colunas : {df.columns.tolist()}")
print()
print(df.head(2))

## 3. Validação da Integridade

In [ ]:
print("── Valores em falta ──")
print(df.isnull().sum())

print(f"\n── Duplicados ──")
print(f"   Notas duplicadas : {df.duplicated(subset='nota_id').sum()}")
print(f"   Textos duplicados: {df.duplicated(subset='nota_texto').sum()}")

print(f"\n── Tipos de nota ──")
print(df['tipo'].value_counts())

print(f"\n── Nível de confiança ──")
print(df['nivel_confianca'].value_counts())

## 4. Binarização de Labels

In [ ]:
# Garantir que labels_lista contém listas Python
df['labels_lista'] = df['labels_lista'].apply(
    lambda x: eval(x) if isinstance(x, str) else x
)

mlb        = MultiLabelBinarizer()
labels_bin = mlb.fit_transform(df['labels_lista'])
label_cols = mlb.classes_.tolist()
df_labels  = pd.DataFrame(labels_bin, columns=label_cols)

print(f"✅ Binarização concluída")
print(f"   Labels identificadas : {len(label_cols)}")
print(f"   Shape binarizado     : {df_labels.shape}")
print()
for i, l in enumerate(label_cols, 1):
    print(f"  {i:2d}. {l}")

## 5. Análise de Distribuição

In [ ]:
freq_abs  = df_labels.sum().sort_values(ascending=False)
freq_rel  = (freq_abs / len(df_labels) * 100).round(1)
train_est = (freq_abs * 0.70).astype(int)

resumo = pd.DataFrame({
    'count'      : freq_abs,
    'pct_%'      : freq_rel,
    'treino_est' : train_est,
    'risco_bert' : freq_abs.apply(
        lambda x: '🔴 CRÍTICO' if x < 20
                  else ('🟡 ALTO' if x < 25 else '🟢 OK')
    )
})

print("── FREQUÊNCIA POR LABEL ──")
print(resumo.to_string())

labels_por_amostra = df_labels.sum(axis=1)
print(f"\n── ESTATÍSTICAS GERAIS ──")
print(f"   Label density média : {labels_por_amostra.mean():.2f} labels/nota")
print(f"   Rácio max/min       : {freq_abs.max()/freq_abs.min():.2f}×")
print(f"   Labels OK  (≥25)    : {(freq_abs >= 25).sum()}/15")
print(f"   Labels Alto (20-24) : {((freq_abs >= 20) & (freq_abs < 25)).sum()}/15")
print(f"   Labels Crítico (<20): {(freq_abs < 20).sum()}/15")

print(f"\n── DISTRIBUIÇÃO DE LABELS POR NOTA ──")
print(labels_por_amostra.value_counts().sort_index().to_string())

## 6. Co-ocorrência (Jaccard)

In [ ]:
mat    = labels_bin
cooc   = mat.T @ mat
n_lbls = len(label_cols)

jaccard = np.zeros((n_lbls, n_lbls))
for i in range(n_lbls):
    for j in range(n_lbls):
        intersection = cooc[i, j]
        union = freq_abs.iloc[i] + freq_abs.iloc[j] - intersection
        jaccard[i, j] = intersection / union if union > 0 else 0

pares = []
for i, j in combinations(range(n_lbls), 2):
    pares.append({
        'label_1' : label_cols[i],
        'label_2' : label_cols[j],
        'n_cooc'  : int(cooc[i, j]),
        'jaccard' : round(float(jaccard[i, j]), 3)
    })

pares_df = pd.DataFrame(pares).sort_values('n_cooc', ascending=False)
print("── TOP 10 CO-OCORRÊNCIAS ──")
print(pares_df.head(10).to_string(index=False))
print(f"\nJaccard máximo: {jaccard[np.triu_indices(n_lbls, k=1)].max():.3f}")
print("→ Co-ocorrência baixa justifica calibração independente por label")

## 7. Visualizações

In [ ]:
os.makedirs('../outputs', exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Frequência por label
colors = ['#ef5350' if c < 20 else '#ffa726' if c < 25 else '#42a5f5'
          for c in freq_abs.values]
freq_abs.plot(kind='barh', ax=axes[0], color=colors)
axes[0].axvline(x=20, color='red',    linestyle='--', alpha=0.7, label='Crítico (<20)')
axes[0].axvline(x=25, color='orange', linestyle='--', alpha=0.7, label='Alto (<25)')
axes[0].set_title('Frequência por Label', fontweight='bold')
axes[0].set_xlabel('Nº de notas')
axes[0].legend(fontsize=8)

# Heatmap co-ocorrência
short = [l.split('(')[0].strip()[:22] for l in label_cols]
mask  = np.eye(n_lbls, dtype=bool)
sns.heatmap(jaccard, xticklabels=short, yticklabels=short,
            ax=axes[1], cmap='Blues', mask=mask, vmin=0, vmax=0.25)
axes[1].set_title('Co-ocorrência (Jaccard)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=7)
axes[1].tick_params(axis='y', labelsize=7)

# Distribuição de labels por nota
dist = labels_por_amostra.value_counts().sort_index()
bars = axes[2].bar(dist.index.astype(str), dist.values, color='#42a5f5')
axes[2].set_title('Nº de Labels por Nota', fontweight='bold')
axes[2].set_xlabel('Nº de diagnósticos')
axes[2].set_ylabel('Nº de notas')
for bar, val in zip(bars, dist.values):
    pct = val / len(df_labels) * 100
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/fig1_distribuicao_dataset.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Figura guardada: outputs/fig1_distribuicao_dataset.png")

## 8. Guardar para Notebooks Seguintes

In [ ]:
os.makedirs('../data', exist_ok=True)

np.save('../data/labels_bin.npy', labels_bin)

with open('../data/label_cols.json', 'w', encoding='utf-8') as f:
    json.dump(label_cols, f, ensure_ascii=False, indent=2)

df.to_csv('../data/notas_processadas.csv', index=False, encoding='utf-8-sig')

print("✅ Dados intermédios guardados:")
print("   data/labels_bin.npy")
print("   data/label_cols.json")
print("   data/notas_processadas.csv")
print()
print("→ Avançar para: 02_model_training.ipynb")